# Notebook 15 — Investigative Case Studies and Lead Quality

**Purpose:** Convert the existing 2024 nonprofit research queue and the Notebook 13 blinded review sample into reproducible, evidence-aware case packets and a structured assessment workflow. **No new fraud model is trained and no ranking is replaced.**

**Important:** A high rank is not a fraud finding, a probability of fraud, or an allegation. Financial anomalies can have legitimate explanations. This notebook prepares materials for qualified reviewers; it does **not** score investigative usefulness without completed human assessments.

**Inputs:** Corrected Notebook 12 primary/limited Parquet exports and Notebook 13 review ID lookup and blinded review worksheet. Optional Notebook 14 ranking comparison is not needed. **Run from the project root or `notebooks/` folder.**

**Confidentiality:** Analyst case packets contain EINs and ranks and are **restricted**. Blinded reviewer packets omit EINs, filing IDs, scores, ranks, model group and diversion labels. Do not publish organization-level suspicion narratives.

## 1. Setup and input validation

In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed'
IN_PRIMARY = DATA / 'notebook12' / '07_2024_organization_primary_queue.parquet'
IN_LIMITED = DATA / 'notebook12' / '07_2024_organization_limited_data_queue.parquet'
IN_LOOKUP = DATA / 'notebook13' / '08_RESTRICTED_review_id_lookup.csv'
IN_WORKSHEET = DATA / 'notebook13' / '08_blinded_investigator_review_template.csv'
OUT = DATA / 'notebook15'
RESTRICTED = OUT / 'restricted_analyst_only'
REVIEWER = OUT / 'blinded_reviewer_packet'
for folder in (OUT, RESTRICTED, REVIEWER):
    folder.mkdir(parents=True, exist_ok=True)
for path in (IN_PRIMARY, IN_LIMITED, IN_LOOKUP, IN_WORKSHEET):
    assert path.is_file(), f'Missing prerequisite: {path}'

def normalize_ein(series):
    z = series.astype('string').str.strip().str.replace(r'^EIN-', '', regex=True)
    z = z.str.replace('-', '', regex=False).str.replace(r'\.0$', '', regex=True).str.zfill(9)
    return z

def number(series):
    return pd.to_numeric(series, errors='coerce').astype('float64').replace([np.inf,-np.inf],np.nan)

p = pd.read_parquet(IN_PRIMARY)
l = pd.read_parquet(IN_LIMITED)
lookup = pd.read_csv(IN_LOOKUP, dtype={'ein':'string','OBJECTID':'string','review_id':'string'})
worksheet = pd.read_csv(IN_WORKSHEET, dtype={'review_id':'string'})
required = {'ein','OBJECTID','tax_year','revenue','assets','expenses','liabilities','net_income',
            'peer_group','six_indicator_count','top_peer_explanations',
            'organization_consensus_rank','organization_supervised_rank','organization_peer_rank'}
assert required.issubset(p.columns), f'Missing primary columns: {sorted(required-set(p.columns))}'
for frame in (p,l):
    frame['ein'] = normalize_ein(frame['ein'])
    frame['OBJECTID'] = frame['OBJECTID'].astype('string')
    assert frame['ein'].str.fullmatch(r'\d{9}').fillna(False).all()
    assert frame.ein.is_unique
    assert frame.tax_year.eq(2024).all()
assert set(p.ein).isdisjoint(set(l.ein))
assert p.organization_consensus_rank.notna().all()
lookup['ein'] = normalize_ein(lookup['ein'])
assert lookup.review_id.is_unique and lookup.ein.is_unique
assert worksheet.review_id.is_unique
assert set(lookup.review_id)==set(worksheet.review_id), 'Review template and lookup IDs differ'
assert set(lookup.ein).issubset(set(p.ein)), 'Some sampled EINs not in the primary queue'
print(f'Primary: {len(p):,}; limited: {len(l):,}; review sample: {len(lookup):,}')
print('Available fields:', len(p.columns))

## 2. Reconcile the review sample without unblinding reviewers

The Notebook 13 restricted lookup is used only inside the analyst environment. **Reviewer-facing files will not contain EINs, OBJECTIDs, ranking positions, model scores, or sampling strata.**

In [ ]:
assert {'review_id','ein','OBJECTID','validation_signal_pattern'}.issubset(lookup.columns)
assert len(lookup)==100, 'Expected 100 cases from Notebook 13; inspect changed sample before proceeding'
matched = lookup.merge(p, on='ein', how='left', validate='one_to_one', suffixes=('_lookup',''))
assert len(matched)==len(lookup)
assert matched['organization_consensus_rank'].notna().all()
assert matched['OBJECTID_lookup'].astype('string').eq(matched['OBJECTID'].astype('string')).all(), 'Filing identifiers changed'
assert 'fraud_label' not in matched.columns, 'Do not distribute outcome labels with case packets'
print('Sample group counts (analyst-only):')
display(matched.validation_signal_pattern.value_counts().rename_axis('group').reset_index(name='cases'))
print('Join integrity: 100% matched, identifiers consistent')

## 3. Document financial trends and peer-relative context

The profiles use only fields already present in Notebook 12. Missing prior-year values remain missing. Percent changes are **not** calculated when the base year is zero or unavailable.

In [ ]:
financial = ['revenue','expenses','assets','liabilities','net_income']
for metric in financial:
    for year in (2022,2023,2024):
        col=f'{metric}_{year}'
        if col in matched.columns:
            matched[col]=number(matched[col])
    if metric in matched.columns:
        matched[metric]=number(matched[metric])
for metric in ['revenue','expenses','assets','liabilities','net_income']:
    a=f'{metric}_2022'; z=f'{metric}_2024'
    if a in matched.columns and z in matched.columns:
        denom=matched[a].abs().where(matched[a].abs()>1e-9)
        matched[f'{metric}_change_2022_2024_pct_review']=100*(matched[z]-matched[a])/denom
        matched[f'{metric}_change_2022_2024_pct_review']=matched[f'{metric}_change_2022_2024_pct_review'].replace([np.inf,-np.inf],np.nan)

peer_metrics=['expense_revenue_ratio','liabilities_assets_ratio','revenue_growth_3yr',
              'asset_growth_3yr','revenue_volatility_3yr','board_independence']
for col in peer_metrics:
    if col in matched.columns:
        matched[col]=number(matched[col])
    pct=f'{col}_peer_percentile'
    if pct in matched.columns:
        matched[pct]=number(matched[pct])

trend_cols=[f'{m}_{y}' for m in financial for y in (2022,2023,2024) if f'{m}_{y}' in matched.columns]
trend_coverage=pd.DataFrame([{'field':col,'available_cases':int(matched[col].notna().sum()),
                               'missing_cases':int(matched[col].isna().sum())} for col in trend_cols])
display(trend_coverage)
trend_coverage.to_csv(OUT/'01_review_sample_history_coverage.csv',index=False)

## 4. Evidence-aware flags and alternative explanations

The flags below are **review prompts**, not determinations. They rely on observed metrics, not assumptions about intent. We do not use a self-reported diversion label as an investigator outcome.

In [ ]:
def build_observations(row):
    observations=[]
    questions=[]
    def val(k):
        x=row.get(k,np.nan)
        return float(x) if pd.notna(x) and np.isfinite(float(x)) else None
    rev=val('revenue'); exp=val('expenses'); assets=val('assets'); liabilities=val('liabilities')
    if rev is not None and rev<=0:
        observations.append('Current-year revenue is nonpositive.')
        questions.append('Verify reporting period, revenue classification, and source filing.')
    if rev is not None and exp is not None and rev>0 and exp>1.25*rev:
        observations.append('Current-year expenses exceed revenue by more than 25%.')
        questions.append('Check reserves, grants, timing differences, and financing of expenditures.')
    if assets is not None and liabilities is not None and assets>0 and liabilities>assets:
        observations.append('Current-year liabilities exceed total assets.')
        questions.append('Check debt structure, capital projects, and accounting classifications.')
    for metric,label in [('revenue','revenue'),('assets','assets')]:
        x=val(f'{metric}_change_2022_2024_pct_review')
        if x is not None and x<=-50:
            observations.append(f'{label.capitalize()} declined at least 50% from 2022 to 2024.')
            questions.append(f'Check one-time events, program changes, and source filings explaining {label} changes.')
    if val('six_indicator_count') is not None and val('six_indicator_count')<6:
        observations.append('At least one peer-anomaly indicator is unavailable.')
        questions.append('Verify whether missing indicators reflect return structure or unavailable source data.')
    peer_text=row.get('top_peer_explanations','')
    if pd.notna(peer_text) and str(peer_text).strip() and str(peer_text)!='No positive adverse peer indicator available':
        observations.append('Peer indicators highlighted: '+str(peer_text))
        questions.append('Review underlying values and peer-group suitability before drawing conclusions.')
    if not observations:
        observations.append('No predefined review-rule threshold triggered; examine peer metrics and historical context.')
    questions.append('Check source Form 990 and supporting schedules before any escalation.')
    return pd.Series({'observed_patterns':' | '.join(dict.fromkeys(observations)),
                      'review_questions':' | '.join(dict.fromkeys(questions)),
                      'possible_nonfraud_explanations':'Grant timing, restricted funding, capital financing, restructuring, accounting/reporting differences, or source-data error; verify case by case.'})

obs=matched.apply(build_observations,axis=1)
matched=pd.concat([matched,obs],axis=1)
assert matched.observed_patterns.notna().all()
display(matched[['review_id','observed_patterns','review_questions']].head(5))

## 5. Generate the **blinded** case-review packet

Reviewer packets include financial facts, peer context and questions, but no organization identifiers, model rankings or selection categories. This reduces score-based anchoring. Source-document access, if authorized, is coordinated separately by a custodian using the restricted lookup.

In [ ]:
review_fields=['review_id','tax_year','peer_group','six_indicator_count',
    'revenue','expenses','assets','liabilities','net_income']
review_fields += trend_cols
review_fields += [c for c in matched.columns if c.endswith('_change_2022_2024_pct_review')]
review_fields += [c for c in peer_metrics if c in matched.columns]
review_fields += [f'{c}_peer_percentile' for c in peer_metrics if f'{c}_peer_percentile' in matched.columns]
review_fields += ['observed_patterns','review_questions','possible_nonfraud_explanations']
review_fields=list(dict.fromkeys(review_fields))
blinded=matched[review_fields].copy().sort_values('review_id').reset_index(drop=True)
for forbidden in ['ein','OBJECTID','organization_consensus_rank','organization_supervised_rank',
                  'organization_peer_rank','consensus_score','supervised_diversion_score',
                  'validation_signal_pattern','fraud_label']:
    assert forbidden not in blinded.columns
assert blinded.review_id.is_unique
blinded.to_csv(REVIEWER/'02_blinded_case_profiles.csv',index=False)
blinded.to_parquet(REVIEWER/'02_blinded_case_profiles.parquet',index=False)
print('Blinded profiles:',len(blinded),'fields:',len(blinded.columns))
display(blinded[['review_id','revenue','assets','observed_patterns']].head(5))

## 6. Analyst-only case profiles and ranking context

These files contain identifiers and rankings. **Keep them restricted** and do not distribute them with the blinded packet. The ranked top-20 preview is separate from the randomized review sample.

In [ ]:
analyst_cols=['review_id','ein','OBJECTID','validation_signal_pattern','peer_group',
    'organization_consensus_rank','organization_supervised_rank','organization_peer_rank',
    'organization_isolation_rank','supervised_diversion_score','peer_adverse_6',
    'consensus_score','six_indicator_count','top_peer_explanations']
analyst_cols=[c for c in analyst_cols if c in matched.columns]
analyst_cols += [c for c in review_fields if c not in analyst_cols and c!='review_id']
analyst=matched[analyst_cols].sort_values('review_id').copy()
analyst.to_parquet(RESTRICTED/'03_RESTRICTED_sample_case_profiles.parquet',index=False)
analyst.to_csv(RESTRICTED/'03_RESTRICTED_sample_case_profiles.csv',index=False)

preview_cols=['ein','OBJECTID','peer_group','revenue','assets','six_indicator_count',
    'organization_consensus_rank','organization_supervised_rank','organization_peer_rank',
    'top_peer_explanations']
preview_cols=[c for c in preview_cols if c in p.columns]
top20=p.nsmallest(20,'organization_consensus_rank')[preview_cols].copy()
top20.to_csv(RESTRICTED/'03_RESTRICTED_top20_combined_preview.csv',index=False)
print('Restricted sample profiles:',len(analyst),'Top-20 preview:',len(top20))
display(analyst[['review_id','validation_signal_pattern','organization_consensus_rank',
                 'organization_supervised_rank','organization_peer_rank']].head(10))

## 7. Structured investigator review worksheet and rubric

No human judgments are prefilled. The rubric separates source verification, alternative explanations, data quality and whether additional records would be useful. A disposition is **not** a criminal-fraud label.

In [ ]:
rubric={
 'objective':'Assess documentary support and usefulness of research leads, not adjudicate fraud.',
 'reviewer_instructions':[
  'Review financial values and historical context without model ranks or sampling groups.',
  'Record source documents actually inspected; do not imply access to unavailable evidence.',
  'Distinguish verified anomalies from possible accounting, funding, or data explanations.',
  'State what additional records would resolve material uncertainty.',
  'Avoid allegations or naming organizations in public outputs.'
 ],
 'review_disposition_options':['no_follow_up','request_additional_records','analyst_escalation','insufficient_information'],
 'evidence_strength_options':['not_assessed','source_unverified','source_verified_only','independently_corroborated'],
 'quality_options':['pass','needs_verification','material_data_issue'],
 'interpretation':'Review dispositions measure analyst judgment, not confirmed fraud or criminal conduct.'
}
(REVIEWER/'04_review_rubric.json').write_text(json.dumps(rubric,indent=2))
review_sheet=worksheet[['review_id']].copy().sort_values('review_id')
fields=['reviewer_id','review_date','source_documents_reviewed','source_verification_status',
        'financial_anomaly_assessment','governance_context','historical_consistency',
        'plausible_nonfraud_explanations','additional_records_needed',
        'independent_corroboration','data_quality_status','evidence_strength',
        'review_disposition','reviewer_notes']
for field in fields:
    review_sheet[field]=''
review_sheet.to_csv(REVIEWER/'04_blank_investigator_review_worksheet.csv',index=False)
print('Blank worksheet:',len(review_sheet),'cases; no outcomes prefilled')

## 8. Lead-quality diagnostics (without inventing review outcomes)

These diagnostics describe the review sample and data coverage. They **cannot** estimate precision, false-positive rate, investigative yield, or fraud detection performance.

In [ ]:
diagnostics={
 'sample_cases':int(len(matched)),
 'primary_population':int(len(p)),
 'limited_population':int(len(l)),
 'matched_sample_cases':int(matched.organization_consensus_rank.notna().sum()),
 'sample_with_six_indicators':int(number(matched.six_indicator_count).eq(6).sum()),
 'sample_with_2022_revenue':int(matched['revenue_2022'].notna().sum()) if 'revenue_2022' in matched else None,
 'sample_with_2023_revenue':int(matched['revenue_2023'].notna().sum()) if 'revenue_2023' in matched else None,
 'sample_with_2024_revenue':int(matched['revenue_2024'].notna().sum()) if 'revenue_2024' in matched else None,
 'human_reviews_completed':0,
 'confirmed_fraud_outcomes_available':False,
 'sample_design':'Up to 25 randomly sampled from each of four model-selection strata; not population-representative',
 'ranking_policy':'Existing Notebook 12 ranks preserved; no retraining or reweighting'
}
(RESTRICTED/'05_RESTRICTED_review_sample_diagnostics.json').write_text(json.dumps(diagnostics,indent=2))
display(pd.DataFrame([diagnostics]).T.rename(columns={0:'value'}))
print('Review sample is deliberately stratified; unweighted disposition rates would not estimate population yield.')

## 9. Optional: summarize **completed** human reviews

This cell **does not assume** reviews have occurred. Once a reviewer completes the worksheet, save a copy as `data/processed/notebook15/blinded_reviewer_packet/04_completed_investigator_reviews.csv`. This cell then validates IDs and dispositions and produces a descriptive summary. No conclusions about fraud are inferred.

In [ ]:
completed_path=REVIEWER/'04_completed_investigator_reviews.csv'
if completed_path.is_file():
    reviews=pd.read_csv(completed_path,dtype={'review_id':'string'})
    assert 'review_id' in reviews and 'review_disposition' in reviews
    assert reviews.review_id.notna().all()
    assert set(reviews.review_id).issubset(set(lookup.review_id))
    allowed=set(rubric['review_disposition_options'])
    dispositions=reviews['review_disposition'].astype('string').str.strip()
    answered=dispositions.notna() & dispositions.ne('')
    invalid=sorted(set(dispositions[answered].dropna())-allowed)
    assert not invalid, f'Unexpected disposition values: {invalid}'
    if 'reviewer_id' in reviews.columns:
        assert not reviews.duplicated(['review_id','reviewer_id']).any(), 'Duplicate case/reviewer assessments'
    else:
        assert reviews.review_id.is_unique, 'Duplicate case assessments without reviewer IDs'
    summary=dispositions[answered].value_counts().rename_axis('disposition').reset_index(name='assessments')
    display(summary)
    summary.to_csv(RESTRICTED/'06_RESTRICTED_review_dispositions_summary.csv',index=False)
    print('Assessments:',int(answered.sum()),'; unique reviewed cases:',reviews.loc[answered,'review_id'].nunique())
    print('These are reviewer dispositions, not verified fraud outcomes.')
else:
    print('No completed reviewer worksheet found. Human-review performance is not yet measurable.')

## 10. Final quality checks and output manifest

In [ ]:
assert len(blinded)==len(lookup)==len(review_sheet)
assert set(blinded.review_id)==set(lookup.review_id)
assert not set(['ein','OBJECTID','validation_signal_pattern','fraud_label']).intersection(blinded.columns)
assert len(analyst)==len(matched)
assert matched['ein'].is_unique
files=[str(f.relative_to(OUT)) for f in OUT.rglob('*') if f.is_file()]
manifest={
 'notebook':'15 — Investigative Case Studies and Lead Quality',
 'inputs':[str(x) for x in (IN_PRIMARY,IN_LIMITED,IN_LOOKUP,IN_WORKSHEET)],
 'sample_cases':int(len(matched)),
 'analyst_restricted_directory':str(RESTRICTED),
 'blinded_reviewer_directory':str(REVIEWER),
 'outputs':sorted(files),
 'limitations':['No independently confirmed fraud outcomes',
                'No investigator dispositions unless completed reviews are supplied',
                'Stratified sample is not population representative',
                'Financial anomaly flags are review prompts, not allegations']
}
(OUT/'07_manifest.json').write_text(json.dumps(manifest,indent=2))
print('QA passed. Outputs in:',OUT)
print('Restricted analyst files and blinded reviewer files are separated.')

## 11. Interpretation and next steps — complete after execution

**What this notebook establishes:** A reproducible process for assembling 100 blinded financial case packets, preserving a restricted identifier/rank mapping, and collecting structured investigator feedback. The financial review prompts are descriptive and should be verified against actual Form 990 documents and any lawfully accessible supporting records.

**What this notebook does not establish:** It does not demonstrate fraud detection accuracy, actionable lead yield, or improved investigative outcomes. No model is retrained, no ranking is promoted, and no organization is labeled fraudulent.

**Next actions:**

1. Confirm sample completeness and inspect any missing history or data-quality issues.
2. Have qualified reviewers assess the blinded cases using authorized source documents, ideally with duplicate independent reviews on a subset.
3. Keep the EIN/OBJECTID lookup restricted to authorized analysts; never distribute it with blinded packets.
4. After completed assessments exist, analyze dispositions by original selection stratum **within the restricted analyst environment**, including inter-reviewer consistency and sensitivity to sample design.
5. Only then decide whether changes to ranking or an investigator-facing product are justified.

**Do not replace this cell with numerical performance claims until human reviews or stronger independent outcome data are available.**